In [2]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from sccoda.util import cell_composition_data as scc_dat
    from sccoda.util import comp_ana as scc_ana
    print("sccoda imported successfully")
except ImportError:
    print("sccoda not installed. Run: pip install sccoda")
    raise

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels_sccoda"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels_sccoda"
for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Same exclusions as original scCODA, applied to cell_type_fine —
# contamination/artefact categories, never real biological populations
EXCLUDE_TYPES_114725 = [
    "Unassigned (n=91, stromal/RBC contamination artefact)",
    "Non-T-cell contamination (from T cells parent cluster)",
]
EXCLUDE_TYPES_176078 = [
    "Unassigned (n=28, doublet/mixed-identity artefact)",
]

print("Setup complete")




sccoda imported successfully
Setup complete


In [2]:
# ----------------------------
# Cell 2 — GSE114725: build composition data, cell_type_fine, Tumour vs Normal
# Reference: "CD4 Naive/Resting T cells" (largest category, resting/
# baseline state, replaces the original "T cells" reference which no
# longer exists as a single category in cell_type_fine).
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")
print(f"Loaded: {adata1.n_obs} cells")

# Drop excluded categories AND the NaN (low-confidence, unresolved) cells
adata1_clean = adata1[
    adata1.obs["cell_type_fine"].notna() &
    ~adata1.obs["cell_type_fine"].isin(EXCLUDE_TYPES_114725)
].copy()
print(f"After exclusions: {adata1_clean.n_obs} cells "
      f"({adata1.n_obs - adata1_clean.n_obs} excluded)")

# Build per-patient, per-tissue composition table
comp_df = adata1_clean.obs.groupby(
    ["patient", "tissue", "cell_type_fine"], observed=True
).size().reset_index(name="count")
comp_wide = comp_df.pivot_table(
    index=["patient", "tissue"], columns="cell_type_fine", values="count", fill_value=0
).reset_index()

print(f"\nComposition table shape: {comp_wide.shape}")
print(comp_wide.head())

# Build scCODA-compatible AnnData
cell_type_cols = [c for c in comp_wide.columns if c not in ["patient", "tissue"]]
sccoda_data_114725 = ad.AnnData(
    X=comp_wide[cell_type_cols].values.astype(float),
    obs=comp_wide[["patient", "tissue"]],
    var=pd.DataFrame(index=cell_type_cols)
)
print(f"\nscCODA AnnData: {sccoda_data_114725.n_obs} samples x {sccoda_data_114725.n_vars} cell types")
print("Reference cell type: CD4 Naive/Resting T cells")

Loaded: 44662 cells
After exclusions: 43381 cells (1281 excluded)


C:\Users\annam\AppData\Local\Temp\ipykernel_20704\2704250254.py:22: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  comp_wide = comp_df.pivot_table(



Composition table shape: (15, 24)
cell_type_fine patient     tissue  Activated CD8 T cells  \
0                  BC1      BLOOD                    0.0   
1                  BC1     NORMAL                  486.0   
2                  BC1      TUMOR                  358.0   
3                  BC2  LYMPHNODE                   82.0   
4                  BC2     NORMAL                  156.0   

cell_type_fine  Antigen-presenting macrophages  B cells  \
0                                         21.0    134.0   
1                                         11.0     12.0   
2                                         53.0    154.0   
3                                         15.0   1233.0   
4                                          9.0     16.0   

cell_type_fine  CD4 Activated T cells  CD4 Naive/Resting T cells  \
0                                30.0                     1286.0   
1                              1319.0                      152.0   
2                               407.0        

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [3]:
# ----------------------------
# Cell 3 — Filter to Tumour vs Normal only, run scCODA
# ----------------------------
tumor_normal_mask = sccoda_data_114725.obs["tissue"].isin(["TUMOR", "NORMAL"])
sccoda_tn = sccoda_data_114725[tumor_normal_mask].copy()
sccoda_tn.obs["tissue"] = sccoda_tn.obs["tissue"].astype(str)

print(f"Filtered to Tumour/Normal: {sccoda_tn.n_obs} samples")
print(sccoda_tn.obs["tissue"].value_counts())

sccoda_model = scc_ana.CompositionalAnalysis(
    sccoda_tn, formula="tissue", reference_cell_type="CD4 Naive/Resting T cells"
)
sccoda_results = sccoda_model.sample_hmc()

print("\n=== GSE114725 cell_type_fine — Tumour vs Normal scCODA results ===")
sccoda_results.summary()

credible = sccoda_results.credible_effects()
print("\nCredible (confidently non-zero) effects:")
print(credible)

sccoda_results.credible_effects().to_csv(
    RESULTS_DIR / "GSE114725_finelabels_scCODA_tumor_vs_normal_credible.csv"
)

Filtered to Tumour/Normal: 12 samples
tissue
TUMOR     8
NORMAL    4
Name: count, dtype: int64
Zero counts encountered in data! Added a pseudocount of 0.5.


100%|██████████| 20000/20000 [03:29<00:00, 95.27it/s] 


MCMC sampling finished. (277.374 sec)
Acceptance rate: 54.1%

=== GSE114725 cell_type_fine — Tumour vs Normal scCODA results ===
Compositional Analysis summary:

Data: 12 samples, 22 cell types
Reference index: 4
Formula: tissue

Intercepts:
                                                    Final Parameter  \
Cell Type                                                             
Activated CD8 T cells                                         1.336   
Antigen-presenting macrophages                                0.126   
B cells                                                       0.021   
CD4 Activated T cells                                         1.387   
CD4 Naive/Resting T cells                                    -0.336   
Complement-high macrophages                                   0.225   
Cycling CD8 T cells                                          -1.085   
Effector CD8 T cells                                          0.017   
LAM-like macrophages                            

In [3]:
# ----------------------------
# Cell 4 — GSE176078: build composition data, cell_type_fine
# Reference: "PVL" — unchanged, same as original scCODA (structural,
# not sub-clustered, not a prime driver candidate).
# ----------------------------
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected_finelabels.h5ad")
print(f"Loaded: {adata2.n_obs} cells")

adata2_clean = adata2[
    adata2.obs["cell_type_fine"].notna() &
    ~adata2.obs["cell_type_fine"].isin(EXCLUDE_TYPES_176078)
].copy()
print(f"After exclusions: {adata2_clean.n_obs} cells "
      f"({adata2.n_obs - adata2_clean.n_obs} excluded)")

comp_df_2 = adata2_clean.obs.groupby(
    ["orig.ident", "subtype", "cell_type_fine"], observed=True
).size().reset_index(name="count")
comp_wide_2 = comp_df_2.pivot_table(
    index=["orig.ident", "subtype"], columns="cell_type_fine", values="count", fill_value=0
).reset_index()

print(f"\nComposition table shape: {comp_wide_2.shape}")

cell_type_cols_2 = [c for c in comp_wide_2.columns if c not in ["orig.ident", "subtype"]]
sccoda_data_176078 = ad.AnnData(
    X=comp_wide_2[cell_type_cols_2].values.astype(float),
    obs=comp_wide_2[["orig.ident", "subtype"]],
    var=pd.DataFrame(index=cell_type_cols_2)
)
sccoda_data_176078.obs["subtype"] = sccoda_data_176078.obs["subtype"].astype(str)
print(f"\nscCODA AnnData: {sccoda_data_176078.n_obs} samples x {sccoda_data_176078.n_vars} cell types")
print("Reference cell type: PVL")

Loaded: 91425 cells
After exclusions: 91397 cells (28 excluded)


C:\Users\annam\AppData\Local\Temp\ipykernel_17192\3462759712.py:19: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  comp_wide_2 = comp_df_2.pivot_table(



Composition table shape: (26, 26)

scCODA AnnData: 26 samples x 24 cell types
Reference cell type: PVL


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [4]:
# ----------------------------
# Cell 5 — Run scCODA for all three pairwise subtype comparisons,
# same loop structure as original notebook 08.
# ----------------------------
pairwise_comparisons = [("TNBC", "ER+"), ("HER2+", "ER+"), ("TNBC", "HER2+")]
all_credible_176078 = {}

for group_a, group_b in pairwise_comparisons:
    comp_name = f"{group_a}_vs_{group_b}"
    mask = sccoda_data_176078.obs["subtype"].isin([group_a, group_b])
    sccoda_sub = sccoda_data_176078[mask].copy()
    sccoda_sub.obs["subtype"] = sccoda_sub.obs["subtype"].astype(str)

    print(f"\n{'='*60}")
    print(f"{comp_name}: {sccoda_sub.n_obs} samples")
    print(f"{'='*60}")

    sccoda_model = scc_ana.CompositionalAnalysis(
        sccoda_sub, formula="subtype", reference_cell_type="PVL"
    )
    sccoda_results = sccoda_model.sample_hmc()
    sccoda_results.summary()

    credible = sccoda_results.credible_effects()
    print(f"\nCredible effects ({comp_name}):")
    print(credible)

    all_credible_176078[comp_name] = credible
    credible.to_csv(RESULTS_DIR / f"GSE176078_finelabels_scCODA_{comp_name}_credible.csv")

print("\n\nGSE176078 fine-label scCODA complete")


TNBC_vs_ER+: 21 samples
Zero counts encountered in data! Added a pseudocount of 0.5.


100%|██████████| 20000/20000 [04:01<00:00, 82.86it/s]


MCMC sampling finished. (309.469 sec)
Acceptance rate: 59.1%
Compositional Analysis summary:

Data: 21 samples, 24 cell types
Reference index: 17
Formula: subtype

Intercepts:
                                       Final Parameter  Expected Sample
Cell Type                                                              
B cells                                         -0.932        96.645904
Basal epithelial                                -1.376        61.995005
CAFs                                            -0.040       235.816468
Cycling T cells                                 -1.132        79.126974
Cycling epithelial                              -0.678       124.593174
Cytotoxic CD8 T cells (reclassified)            -0.997        90.563732
Effector CD8 T cells                            -0.789       111.503260
Endothelial cells                                0.204       300.983012
Epithelial (ambiguous)                          -0.715       120.067469
Exhausted CD8 T cells           

100%|██████████| 20000/20000 [03:37<00:00, 91.88it/s] 


MCMC sampling finished. (278.640 sec)
Acceptance rate: 64.6%
Compositional Analysis summary:

Data: 16 samples, 24 cell types
Reference index: 17
Formula: subtype

Intercepts:
                                       Final Parameter  Expected Sample
Cell Type                                                              
B cells                                         -0.736        99.897738
Basal epithelial                                -1.412        50.812736
CAFs                                            -0.047       198.968603
Cycling T cells                                 -1.199        62.874900
Cycling epithelial                              -0.791        94.551725
Cytotoxic CD8 T cells (reclassified)            -0.920        83.108505
Effector CD8 T cells                            -0.765        97.042307
Endothelial cells                                0.439       323.483154
Epithelial (ambiguous)                          -0.872        87.195004
Exhausted CD8 T cells           

100%|██████████| 20000/20000 [03:41<00:00, 90.20it/s] 


MCMC sampling finished. (282.431 sec)
Acceptance rate: 61.6%
Compositional Analysis summary:

Data: 15 samples, 24 cell types
Reference index: 17
Formula: subtype

Intercepts:
                                       Final Parameter  Expected Sample
Cell Type                                                              
B cells                                         -0.478       116.285635
Basal epithelial                                -1.238        54.382888
CAFs                                             0.264       244.215135
Cycling T cells                                 -0.782        85.802620
Cycling epithelial                              -0.518       111.726010
Cytotoxic CD8 T cells (reclassified)            -0.620       100.891889
Effector CD8 T cells                            -0.511       112.510836
Endothelial cells                               -0.031       181.825882
Epithelial (ambiguous)                          -0.335       134.162203
Exhausted CD8 T cells           